In [ ]:
import io
import tempfile
import zipfile
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyarrow as pa
from tqdm import tqdm

from paths import paths, prepare_preprocessing
from preprocessing.src.constants import CRS_METRIC, LSOA_IDS, REGION
from preprocessing.src.geo import get_buffered_region, intersecting_features
from preprocessing.src.isochrone import extract_zones_by_duration
from preprocessing.src.threading import parallelize

In [ ]:
prepare_preprocessing()

In [ ]:
EPC_CERTIFICATES_CSV = paths.raw.walkability.epc
PARQUET_DWELLINGS_EPC = paths.preprocessing.shared.epc
PARQUET_DWELLINGS_IN_REGION = paths.preprocessing.shared.dwellings

BUILT_FORM_CATEGORIES = {
    'Mid-Terrace': 'mid_terrace',
    'Semi-Detached': 'semi_detached',
    'End-Terrace': 'end_terrace',
    'Detached': 'detached',
    'Enclosed End-Terrace': 'enclosed_end_terrace',
    'Enclosed Mid-Terrace': 'enclosed_mid_terrace',
    'NO DATA!': 'no_data'
}

In [ ]:
GENERATE_MISSING_DATA = not PARQUET_DWELLINGS_IN_REGION.exists()

In [ ]:
# If the raw EPC data has not been processed, process it, else fetch it from disk
if GENERATE_MISSING_DATA:
    region = get_buffered_region(REGION, buffer = 5000)
    os_uprn_gdf = gpd.read_file(paths.raw.walkability.uprn, engine = 'fiona')

    chunk_size = 100_000
    cols = ['BUILT_FORM', 'UPRN']

    # The raw data is too large to fit into memory, so it must be processed in chunks
    with tempfile.TemporaryDirectory(prefix = 'pq_chunks_') as tmp:
        tmp = Path(tmp)

        with zipfile.ZipFile(EPC_CERTIFICATES_CSV) as z:
            with z.open('epc_certificates.csv') as f:
                n_lines = sum(1 for _ in f) - 1

            n_chunks = (n_lines // chunk_size) + 1

            for i, chunk in enumerate(
                tqdm(
                    pd.read_csv(z.open('epc_certificates.csv'), chunksize = chunk_size, low_memory = False),
                    total = n_chunks,
                    desc = 'Reading CSV in chunks',
                )
            ):
                out = tmp / f'chunk_{i:04d}.parquet'
                chunk.to_parquet(out)

        parquet_files = sorted(tmp.glob('*.parquet'))

        tables = []

        for p in tqdm(parquet_files, desc = 'Reading parquet chunks'):
            table = pa.parquet.read_table(str(p), columns = cols)
            tables.append(table)

        combined = pa.concat_tables(tables, unify_schemas = True)
        pa.parquet.write_table(combined, str(PARQUET_DWELLINGS_EPC), compression = 'snappy')

    df = pd.read_parquet(PARQUET_DWELLINGS_EPC)
    addresses_joined = os_uprn_gdf.merge(df, on = 'UPRN', how = 'inner')
    addresses_in_area = addresses_joined[addresses_joined.intersects(region)]

    addresses_in_area.to_parquet(PARQUET_DWELLINGS_IN_REGION, index = False)

else:
    addresses_in_area = gpd.read_parquet(PARQUET_DWELLINGS_IN_REGION)

In [ ]:
def add_dwelling_score(gdf):
    gdf['dwelling_score'] = (gdf['dwelling_count_per_km2'] - gdf['dwelling_count_per_km2'].mean()) / gdf[
        'dwelling_count_per_km2'
    ].std(ddof = 0)

    return gdf

def process_file(lsoa_id):
    candidates = {}
    results = {}

    for minutes, zones in extract_zones_by_duration(lsoa_id, buffer = 15, crs = CRS_METRIC).items():
        densities = []
        fractions_list = []

        for seed, zone in zones:
            addresses_in_zone = intersecting_features(addresses_in_area, zone, candidates.get(seed))
            candidates[seed] = addresses_in_zone

            n = len(addresses_in_zone)
            area_km2 = zone.area / 1e6
            densities.append(n / area_km2 if area_km2 > 0 else 0)

            counts = (
                addresses_in_zone['BUILT_FORM']
                .value_counts()
                .reindex(BUILT_FORM_CATEGORIES.keys(), fill_value = 0)
                .rename(BUILT_FORM_CATEGORIES)
            )

            fracs = counts / n if n > 0 else pd.Series(0, index = counts.index)
            fractions_list.append(fracs)

        results[minutes] = {
            'lsoa_id': lsoa_id,
            'dwelling_count_per_km2': np.mean(densities),
            'dwelling_built_form_fractions': pd.DataFrame(fractions_list).mean().to_dict()
        }

    return results

# Build the shared spatial index before starting workers (avoid forking native geospatial libraries)
addresses_in_area.sindex

results = parallelize(process_file, LSOA_IDS, max_workers = 4, use_threads = True)

In [ ]:
lsoa_base = gpd.read_parquet(paths.preprocessing.shared.lsoa)
results_by_duration = {minutes: [result[minutes] for result in results] for minutes in (20, 15, 10)}

In [ ]:
component_data = {}

for minutes, duration_results in results_by_duration.items():
    addresses_per_sq_km = {
        result['lsoa_id']: result['dwelling_count_per_km2'] for result in duration_results
    }
    addresses_type_fractions = {
        result['lsoa_id']: result['dwelling_built_form_fractions'] for result in duration_results
    }
    
    # Load the LSOA data and add the dwelling score, counts, and fractions
    lsoa = lsoa_base.copy()
    lsoa['dwelling_count_per_km2'] = lsoa['LSOA21CD'].map(addresses_per_sq_km).fillna(0).astype(int)

    for category in BUILT_FORM_CATEGORIES.values():
        lsoa[f'dwelling_fraction_{category}'] = (
            lsoa['LSOA21CD'].map(lambda x: addresses_type_fractions.get(x, {}).get(category, 0)).fillna(0).astype(float)
        )

    lsoa = add_dwelling_score(lsoa)

    columns = [
        'LSOA21CD',
        'geometry',
        'dwelling_score',
        'dwelling_count_per_km2',
        *[f'dwelling_fraction_{category}' for category in BUILT_FORM_CATEGORIES.values()]
    ]
    
    lsoa[columns].to_parquet(paths.preprocessing.walkability[minutes].dwellings, index = False)
    component_data[minutes] = lsoa

In [ ]:
for minutes, lsoa in component_data.items():
    FILES = paths.preprocessing.walkability[minutes]
    
    m = lsoa.explore(
        column = 'dwelling_score',
        cmap = 'viridis',
        scheme = 'quantiles',
        k = 50,
        legend = True,
        tiles = 'CartoDB positron',
        style_kwds = {'weight': 0.5, 'lineOpacity': 0.5, 'fillOpacity': 0.375},
    )

    m.save(FILES.dwellings_map)

m